# 01. Exploratory Data Analysis & Data Preprocessing

**Lead Responsible**: Pehesara W.A.C (`IT24103684`)  
**Module**: IT3091 – Machine Learning | Group 2026-AI-15  
**Track**: Industry Explorer Track  

---

## Objectives
1. Ingest the Kaggle E-Commerce 2023 proxy dataset and inspect schema structure.
2. Handle missing values (e.g. `days_since_prior_order` for first orders) and duplicates.
3. Analyze basket size distributions, product co-occurrence frequencies, and department breakdowns.
4. Export cleaned, optimized parquet files into `data/processed/`.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import clean_ecommerce_data, create_stratified_sample
from src.utils import setup_plot_style, save_figure

setup_plot_style()
print("Environment initialized successfully.")

Environment initialized successfully.


In [2]:
import sys
from pathlib import Path

# Resolve project root dynamically (works in IDE and JupyterLab)
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Core data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & scaling modules from scikit-learn
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

# Project utilities
from src.data_loader import load_raw_dataset
from src.utils import setup_plot_style, save_figure, export_summary_table

# Set consistent publication theme
setup_plot_style()

print(f"[OK] Python Interpreter : {sys.executable}")
print(f"[OK] Project Root       : {PROJECT_ROOT}")

[OK] Python Interpreter : E:\Git_Projects\Supermarket\.venv\Scripts\python.exe
[OK] Project Root       : E:\Git_Projects\Supermarket


## 1. Data Ingestion & Diagnostic Health Audit
We load the raw transactional log (`ECommerce_consumer_behaviour.csv`) and compute key health indicators:
row counts, attribute types, memory footprint, and distinct entity cardinalities.

In [5]:
RAW_FILE = "ECommerce_consumer_behaviour.csv"
df_raw = load_raw_dataset(RAW_FILE)

total_rows, total_cols = df_raw.shape
initial_mem = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)

print("DATASET INTEGRITY & HEALTH DIAGNOSTIC")
print(f"Total Transaction Rows : {total_rows:,}")
print(f"Total Column Features  : {total_cols}")
print(f"Initial Memory In RAM  : {initial_mem:.2f} MB")
print("Column Cardinalities & Types:")
for col in df_raw.columns:
    print(f"  - {col:<24}: {df_raw[col].nunique():>8,} unique | {df_raw[col].dtype}")

[Loading Dataset] -> ECommerce_consumer_behaviour.csv (Source: E:\Git_Projects\Supermarket\data\raw)
DATASET INTEGRITY & HEALTH DIAGNOSTIC
Total Transaction Rows : 2,019,501
Total Column Features  : 12
Initial Memory In RAM  : 228.11 MB
Column Cardinalities & Types:
  - order_id                :  200,000 unique | int64
  - user_id                 :  105,273 unique | int64
  - order_number            :      100 unique | int64
  - order_dow               :        7 unique | int64
  - order_hour_of_day       :       24 unique | int64
  - days_since_prior_order  :       31 unique | float64
  - product_id              :      134 unique | int64
  - add_to_cart_order       :      137 unique | int64
  - reordered               :        2 unique | int64
  - department_id           :       21 unique | int64
  - department              :       21 unique | str
  - product_name            :      134 unique | str


## 2. Duplicate Detection & Granular Integrity

In [6]:
# 1. Full row duplicates
full_dups = df_raw.duplicated().sum()

# 2. Duplicate product within the same order (order_id + product_id)
intra_order_dups = df_raw.duplicated(subset=["order_id", "product_id"]).sum()

# 3. Duplicate cart position in the same order (order_id + add_to_cart_order)
cart_index_collisions = df_raw.duplicated(subset=["order_id", "add_to_cart_order"]).sum()

print("--- Duplicate Audit Results ---")
print(f"1. Exact Full-Row Duplicates                 : {full_dups}")
print(f"2. Intra-Order Product Duplicates (Repeats)  : {intra_order_dups:,} ({(intra_order_dups/len(df_raw))*100:.2f}%)")
print(f"3. Cart Index Position Collisions            : {cart_index_collisions}")

# If full duplicates existed, we would execute:
if full_dups > 0:
    df_raw = df_raw.drop_duplicates()
    print(f"[ACTION] Dropped {full_dups} duplicate rows.")
else:
    print("[STATUS] No exact full-row duplicates detected.")

--- Duplicate Audit Results ---
1. Exact Full-Row Duplicates                 : 0
2. Intra-Order Product Duplicates (Repeats)  : 566,385 (28.05%)
3. Cart Index Position Collisions            : 0
[STATUS] No exact full-row duplicates detected.


## 3. Structural Missing Value Diagnostic
An audit of null values reveals that missing values only exist in `days_since_prior_order`.  
We test the hypothesis that missing values are **Missing Not At Random (MNAR)** and correspond precisely to each customer's initial acquisition order (`order_number == 1`).

In [7]:
null_counts = df_raw.isnull().sum()
print("Null Values by Attribute:")
print(null_counts[null_counts > 0])

# Verify that nulls only occur when order_number == 1
first_order_nulls = df_raw[df_raw["order_number"] == 1]["days_since_prior_order"].isna().sum()
subsequent_nulls = df_raw[df_raw["order_number"] > 1]["days_since_prior_order"].isna().sum()
total_nulls = df_raw["days_since_prior_order"].isna().sum()

print("\n--- Structural Null Verification ---")
print(f"Total nulls in 'days_since_prior_order' : {total_nulls:,}")
print(f"Nulls on initial orders (order_number=1): {first_order_nulls:,} ({(first_order_nulls/total_nulls)*100:.1f}%)")
print(f"Nulls on subsequent repeat orders       : {subsequent_nulls:,} (0.0%)")

Null Values by Attribute:
days_since_prior_order    124342
dtype: int64

--- Structural Null Verification ---
Total nulls in 'days_since_prior_order' : 124,342
Nulls on initial orders (order_number=1): 124,342 (100.0%)
Nulls on subsequent repeat orders       : 0 (0.0%)


In [8]:
# Academic proof
assert first_order_nulls == total_nulls, "Anomaly: missing values found in repeat orders!"
print("\n[VALIDATED] All nulls are structurally explained by initial baseline orders.")

# Treatment: Flag first orders and impute baseline with sentinel value (-1)
df_clean = df_raw.copy()
df_clean["is_first_order"] = df_clean["days_since_prior_order"].isna().astype("bool")
df_clean["days_since_prior_order"] = df_clean["days_since_prior_order"].fillna(-1)


[VALIDATED] All nulls are structurally explained by initial baseline orders.


## 4. Outlier Analysis & Extreme Basket Detection (IQR Method)

In [9]:
# Compute items per basket
df_clean["basket_size"] = (
    df_clean.groupby("order_id", observed=False)["product_id"]
    .transform("count")
)

# Extract unique order basket sizes
order_baskets = df_clean.drop_duplicates(subset=["order_id"])["basket_size"]

# IQR calculations
q1 = order_baskets.quantile(0.25)
q3 = order_baskets.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

outlier_orders = order_baskets[order_baskets > upper_bound]
pct_outliers = (len(outlier_orders) / len(order_baskets)) * 100

print("--- Basket Size Outlier Audit ---")
print(f"25th Percentile (Q1)     : {q1:.1f} items")
print(f"Median (50th Percentile) : {order_baskets.median():.1f} items")
print(f"75th Percentile (Q3)     : {q3:.1f} items")
print(f"IQR                      : {iqr:.1f} items")
print(f"Outlier Threshold        : > {upper_bound:.1f} items")
print(f"Outlier Orders Count     : {len(outlier_orders):,} ({pct_outliers:.2f}% of orders)")

--- Basket Size Outlier Audit ---
25th Percentile (Q1)     : 5.0 items
Median (50th Percentile) : 8.0 items
75th Percentile (Q3)     : 14.0 items
IQR                      : 9.0 items
Outlier Threshold        : > 27.5 items
Outlier Orders Count     : 6,589 (3.29% of orders)
